# Mecanismes d Attention des Transformateurs

## Contexte et Objectifs

Ce notebook est un guide approfondi sur le mecanisme d'attention, qui est au cœur de l'architecture des Transformateurs, une revolution dans le domaine du traitement du langage naturel (NLP) et au-dela. Comprendre l'attention est essentiel pour saisir comment les modeles comme BERT, GPT et T5 fonctionnent.

Nous allons implementer les composants cles de l'attention a partir de zero en utilisant PyTorch pour une comprehension detaillee et intuitive.

### Concepts Cles Abordes :

1.  **L'Idee Fondamentale de l'Attention :** Nous expliquerons comment l'attention permet a un modele de se concentrer sur les parties les plus pertinentes de l'entree lors du traitement d'une sequence. Nous introduirons les concepts de **Requetes (Queries)**, **Cles (Keys)**, et **Valeurs (Values)**.
2.  **Attention a Produit Scalaire (Scaled Dot-Product Attention) :** C'est la forme d'attention la plus courante. Nous l'implementerons et visualiserons la matrice d'attention pour voir comment les mots interagissent entre eux.
3.  **Attention Multi-Tetes (Multi-Head Attention) :** Une technique qui permet au modele d'apprendre differentes relations en parallele en utilisant plusieurs "tetes d'attention". Nous construirons ce mecanisme en encapsulant plusieurs couches d'attention a produit scalaire.
4.  **Couche d'Encodeur de Transformateur :** Nous assemblerons une couche complete d'encodeur, qui combine l'attention multi-tetes avec un reseau de neurones a propagation avant (Feed-Forward), des connexions residuelles et une normalisation de couche.
5.  **Assemblege d'un Encodeur Simple :** Nous montrerons comment empiler plusieurs couches d'encodeur pour former le bloc principal d'un Transformateur.

_Derniere mise a jour : 2026-02-16_

In [1]:
# --- 1. Installation des Dependances ---
%pip install -q torch numpy matplotlib
print("Dependances installees.")

✅ Output snapshot saved (sanitized): execution artifacts prepared for GitHub rendering.\n

In [2]:
# --- 2. Imports et Configuration ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import logging

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

✅ Output snapshot saved (sanitized): execution artifacts prepared for GitHub rendering.\n

## 3. Implementation de l'Attention a Produit Scalaire

L'attention est calculee selon la formule : `Attention(Q, K, V) = softmax(Q * K^T / sqrt(d_k)) * V`

In [3]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """Calcule l'attention a produit scalaire."""
    d_k = query.size(-1)  # Dimension des cles
    scores = torch.matmul(query, key.transpose(-2, -1)) / np.sqrt(d_k)

    # Appliquer le masque (si fourni, pour le decodage par exemple)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, value)
    return output, attention_weights

# --- Exemple d'Utilisation ---
# Supposons une sequence de 3 mots, chacun avec un embedding de dimension 4
seq_len, d_k = 3, 4
query = torch.randn(seq_len, d_k)
key = torch.randn(seq_len, d_k)
value = torch.randn(seq_len, d_k)

output, attention_weights = scaled_dot_product_attention(query, key, value)

logger.info("Poids de l'attention (matrice 3x3) :")
print(attention_weights)

# Visualisation des poids
plt.figure(figsize=(5, 5))
plt.imshow(attention_weights.detach().numpy(), cmap='viridis')
plt.title("Matrice d'Attention")
plt.xlabel("Cles")
plt.ylabel("Requetes")
plt.colorbar()
plt.show()

✅ Output snapshot saved (sanitized): execution artifacts prepared for GitHub rendering.\n

## 4. Implementation de l'Attention Multi-Tetes

L'attention multi-tetes projette les requetes, cles et valeurs dans differents sous-espaces pour capturer diverses relations.

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.d_model = d_model
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def split_heads(self, x):
        batch_size = x.size(0)
        return x.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
    
    def forward(self, query, key, value, mask=None):
        # Projections lineaires
        q = self.W_q(query)
        k = self.W_k(key)
        v = self.W_v(value)
        
        # Diviser en tetes multiples
        q = self.split_heads(q) # (batch_size, num_heads, seq_len, d_k)
        k = self.split_heads(k) # (batch_size, num_heads, seq_len, d_k)
        v = self.split_heads(v) # (batch_size, num_heads, seq_len, d_k)
        
        # Calculer l'attention
        attention_output, attention_weights = scaled_dot_product_attention(q, k, v, mask)
        
        # Concatener les tetes et appliquer la projection finale
        attention_output = attention_output.transpose(1, 2).contiguous().view(query.size(0), -1, self.d_model)
        output = self.W_o(attention_output)
        return output

# --- Exemple ---
d_model = 128
num_heads = 8
batch_size = 32
seq_length = 10

multi_head_attention = MultiHeadAttention(d_model, num_heads)
q = torch.randn(batch_size, seq_length, d_model)
k = torch.randn(batch_size, seq_length, d_model)
v = torch.randn(batch_size, seq_length, d_model)

output = multi_head_attention(q, k, v)
logger.info(f"Forme de la sortie de l'attention multi-tetes : {output.shape}")

✅ Output snapshot saved (sanitized): execution artifacts prepared for GitHub rendering.\n

## 5. Couche d'Encodeur de Transformateur

Une couche d'encodeur combine l'attention multi-tetes avec un reseau feed-forward, en utilisant des connexions residuelles et la normalisation de couche.

In [5]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # Sous-couche d'attention
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Sous-couche feed-forward
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

# --- Exemple ---
d_ff = 512 # Dimension du reseau feed-forward
encoder_layer = EncoderLayer(d_model, num_heads, d_ff)

input_tensor = torch.randn(batch_size, seq_length, d_model)
output_tensor = encoder_layer(input_tensor)
logger.info(f"Forme de la sortie de la couche d'encodeur : {output_tensor.shape}")

✅ Output snapshot saved (sanitized): execution artifacts prepared for GitHub rendering.\n

## 6. Construction d'un Encodeur de Transformateur Complet

Un encodeur complet est simplement une pile de N couches d'encodeur identiques.

In [6]:
class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super(TransformerEncoder, self).__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return x

# --- Exemple avec un encodeur a 6 couches ---
num_layers = 6
encoder = TransformerEncoder(num_layers, d_model, num_heads, d_ff)

final_output = encoder(input_tensor)
logger.info(f"Forme de la sortie de l'encodeur complet : {final_output.shape}")

✅ Output snapshot saved (sanitized): execution artifacts prepared for GitHub rendering.\n